In [1]:
# Import necessary libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import LambdaCDM
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Set cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "bold",
    'font.size': 25,
    'font.weight': 'normal',
    
    # Tick direction and appearance
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,            # show top ticks
    'ytick.right': True,          # show right ticks
    'xtick.minor.visible': True,  # show minor x ticks
    'ytick.minor.visible': True,  # show minor y ticks
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    
    # Axes and line properties
    'lines.linewidth': 2,
    'axes.linewidth': 3.5,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})
# Pandas configuration
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')
df['p_modelmag_u_0'] = df['p_modelmag_u'] - df['p_extinction_u']
df['p_modelmag_g_0'] = df['p_modelmag_g'] - df['p_extinction_g']
df['p_modelmag_r_0'] = df['p_modelmag_r'] - df['p_extinction_r']
df['p_modelmag_i_0'] = df['p_modelmag_i'] - df['p_extinction_i']
df['p_modelmag_z_0'] = df['p_modelmag_z'] - df['p_extinction_z']

df['p_petromag_u_0'] = df['p_petromag_u'] - df['p_extinction_u']
df['p_petromag_g_0'] = df['p_petromag_g'] - df['p_extinction_g']
df['p_petromag_r_0'] = df['p_petromag_r'] - df['p_extinction_r']
df['p_petromag_i_0'] = df['p_petromag_i'] - df['p_extinction_i']  
df['p_petromag_z_0'] = df['p_petromag_z'] - df['p_extinction_z']  
df['grmod'] = df['p_modelmag_g_0'] - df['p_modelmag_r_0']

In [47]:
pseudo_member_df = df[df['z_tot_z'].between(0.03-0.01, 0.03+0.01)]

In [48]:
cond = (pseudo_member_df['p_petromag_r']-pseudo_member_df['p_fibermag_r'])<-1
target = pseudo_member_df[cond]

In [49]:
# Open a text file and write the header + data manually
with open('06b_petromag_validity_check_SDSSimglist_input.txt', 'w') as f:
    # f.write("index TARGET_RA TARGET_DEC\n")
    temp = target.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")


In [50]:
# Open a text file and write the header + data manually
with open('06c_petromag_validity_check_SDSSimglist_result.txt', 'w') as f:
    f.write("objid,petromag_r,fibermag_r,Flag\n")
    f.write("Flag 0 = Invalid Petrosian (Use fiber mag), 1 = Valid Petrosian\n")
    temp = target.sort_values(by='p_petromag_r', ascending=True)
    
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']},{row['p_ra']:.6f},{row['p_dec']:.6f},{row['p_petromag_r']:.2f},{row['p_fibermag_r']:.2f},1\n")

A2199 Members에 대해서 SDSS imglist 작업을 visual로 진행

A2199 redshift = 0.03, Caustic 돌리기전이니까 0.03 \pm 0.01 정도만의 target 대상으로 redshfit 대상 선택해서 visual inspection 진행할 수 있음

1. petrosian magnitude 밝은 쪽에서 어두운쪽으로 order해서 0.1''/pixel로 이미지 관찰 
2. 주변 petrosian magnitude order에서 벗어난 (특히 더 어두워서) fiber magnitude 써야할 것 같은 녀석들 list 

비록 Redshift가 측정된 Target이라서 Galaxy라는 것이 이번 분석 과정에서 확실히 단정되었다고 하여도 그 Photometry가 Invalid하여 p_petromag_r을 사용하는 것이 신뢰할 수 없는 값인 경우가 있다. 이 경우에는 p_petromag_r을 p_fibermag_r로 교체해서 진행한다. 

아래의 objid list는 Redshift가 측정된 Target 중에서 SDSS imgliist로 visual classification을 진행해본 결과, photometry가 invalid한 candidate 후보군 들이다. (Nearby bright source, faint in image than fiber magnitude... 등이 있다.)

In [ ]:
gal_fibermag_a2199members_objid_list = [
    1237659330315223520, 
    1237659330315289013,
    1237659326566171345,
    1237659330852356638,
    1237659330852094660,
    # 1237659330315485821,
    # 1237659325492495126,
    # 1237659330315485760,
    # 1237655471820833890
]



In [51]:
import pandas as pd

file_path = "./06c_petromag_validity_check_SDSSimglist_result.txt"

# Row 1 is a descriptive note, so skip it and assign column names explicitly.
vis = pd.read_csv(
    file_path,
    sep=r"\s*,\s*",
    engine="python",
    skiprows=2,
    names=["objid", "RA", "DEC", "petromag_r", "fibermag_r", "Flag"],
)

vis["Flag"] = vis["Flag"].astype(int)
vis.to_csv("./06d_petromag_validity_check_SDSSimglist_result.csv", index=False)

해당하는 녀석들 중에서 'MEMBER=Y' + petromag가 fibermag보다 1등급이상 밝은 녀석들을 골랐다. 이 녀석들은 nearby bright source에 의해서 실제로 어두운데 밝게 측정된 것이다. 즉, 주어진 SDSS img를 보았을 때 target galaxy의 magnitude가 petromag보다 더 어두워보여서 (주변 source의 오염에 의해 더 밝게 측정?) fiber magnitude를 활용하여 바꿔치기를 진행하고자 한다. 